In [36]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("IME")
    .master("local[*]")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "4g")
    .config("spark.ui.port", "4040")
    .config("spark.port.maxRetries", "0")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark", spark.version, "prêt.")

Spark 3.5.3 prêt.


### Pourquoi `spark.driver.memory` à 4g
En `local[*]`, tout tourne dans un seul processus JVM : le driver fait aussi le travail des executors. Par défaut, Spark ne lui donne qu'1 Go. Nos quatre mois de 2025 représentent 15,7 millions de trajets : les fichiers Parquet sont compressés sur le disque, mais ils se déplient beaucoup en mémoire, et des opérations comme `dropDuplicates` (un shuffle sur toutes les colonnes) ont besoin de place. Avec 1 Go, on risque des erreurs `OutOfMemory` ou des débordements massifs sur le disque. 4 Go laissent assez de marge tout en restant raisonnable pour un ordinateur portable.

# Séance 1

## Étape 1 : Environnement

- Projet `~/tp_spark_core` avec l'arborescence `data/{raw,bronze,silver,gold,quarantine}` et `notebooks/`. Le notebook tourne depuis `notebooks/`, donc tous les chemins commencent par `../data/`.
- Versions figées dans `requirements.txt` (`pyspark==3.5.3`, `delta-spark==3.2.1`, …) : on ne laisse jamais pip choisir une version.
- Mémoire du driver à 4g : voir la justification juste après la cellule de démarrage.
- `spark.sql.shuffle.partitions = 4` : la valeur par défaut (200) est pensée pour un cluster ; en local, 200 petites tâches coûtent plus en coordination qu'elles ne rapportent.

## Étape 3 : Investigation (Data Discovery)

In [37]:
mois = ["01", "04", "07", "10"]

for m in mois:
    path = f"../data/raw/yellow_tripdata_2025-{m}.parquet"
    df_discovery = spark.read.format("parquet").load(path)
    print(f"===== Mois {m} : {len(df_discovery.columns)} colonnes =====")
    df_discovery.printSchema()

===== Mois 01 : 20 colonnes =====
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

===== Mois 04 : 20 colonnes =====
root
 |

In [38]:
df_discovery = spark.read.format("parquet").load("../data/raw/yellow_tripdata_2025-01.parquet")
df_discovery.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

### Question 1 : les quatre fichiers ont-ils exactement le même schéma ?
**Oui.** Les quatre fichiers de 2025 (janvier, avril, juillet, octobre) ont exactement le même schéma : 20 colonnes, mêmes noms, mêmes types, toutes `nullable`.

**Comment je l'ai vérifié :** j'ai lu chaque fichier séparément avec l'inférence de schéma et affiché son `printSchema()` ainsi que son nombre de colonnes (20 pour chaque mois), puis j'ai comparé les quatre sorties colonne par colonne. La cellule suivante automatise cette comparaison : elle construit pour chaque mois un dictionnaire `{colonne: type}` et affiche toute colonne absente ou de type différent ; elle ne signale aucune différence. Les types relevés sont les suivants :
- `VendorID`, `PULocationID`, `DOLocationID` : `integer` ;
- `passenger_count`, `RatecodeID`, `payment_type` : `long` (et non `double`) ;
- `trip_distance` et tous les montants : `double` ;
- `tpep_pickup_datetime`, `tpep_dropoff_datetime` : `timestamp_ntz` (heure locale de New York, sans fuseau) ;
- `store_and_fwd_flag` : `string`.

Mon `StructType` n'a donc pas à gérer de variation entre les mois : un seul schéma couvre les quatre fichiers. À noter, la colonne `cbd_congestion_fee` (taxe de congestion de Manhattan, introduite en janvier 2025) est présente dans les quatre fichiers. Elle n'existe pas dans les années précédentes, ce qui montre que la TLC fait évoluer la structure de ses fichiers d'une année à l'autre. C'est pour cette raison qu'il faut vérifier chaque mois et chaque année au lieu de supposer un schéma fixe.

In [39]:
# Comparaison automatique des schémas des 4 mois
schemas = {}
for m in mois:
    df = spark.read.format("parquet").load(f"../data/raw/yellow_tripdata_2025-{m}.parquet")
    schemas[m] = {f.name: f.dataType.simpleString() for f in df.schema.fields}

toutes_colonnes = sorted(set().union(*[s.keys() for s in schemas.values()]))

differences = False
for col_name in toutes_colonnes:
    types = {m: schemas[m].get(col_name, "ABSENTE") for m in mois}
    if len(set(types.values())) > 1:
        differences = True
        print(f"{col_name} : {types}")

if not differences:
    print("Les 4 mois ont exactement les mêmes colonnes et les mêmes types.")

Les 4 mois ont exactement les mêmes colonnes et les mêmes types.


## Étape 4 : Ingestion Robuste (Bronze)

- **Consigne A** : schéma manuel strict (`StructType`), construit à partir de l'investigation de l'étape 3. Les 4 mois de 2025 ont les mêmes 20 colonnes et les mêmes types, donc un seul schéma suffit. Attention : `passenger_count`, `RatecodeID` et `payment_type` sont en `long` dans les fichiers 2025 (et non en `double`), il faut les déclarer en `LongType` sinon la lecture plante avec une erreur de conversion Parquet. Les dates restent en `TimestampNTZType` (heures locales de New York, sans fuseau).
- **Consigne B** : ajout des deux colonnes de traçabilité `ingestion_date` (`current_timestamp()`) et `source_file` (`input_file_name()`).
- Écriture au format Delta, par chemin, dans `../data/bronze/bronze_trips` (pas de `saveAsTable` en local).

In [40]:
from pyspark.sql.types import (StructType, StructField, IntegerType, LongType,
                               StringType, TimestampNTZType, DoubleType)
from pyspark.sql.functions import current_timestamp, input_file_name

# 1. Dossier local des fichiers téléchargés
raw_path = "../data/raw"

# 2. Schéma manuel (contrat strict), identique pour les 4 mois de 2025
taxi_schema = StructType([
    StructField("VendorID", IntegerType(), True),
    StructField("tpep_pickup_datetime", TimestampNTZType(), True),
    StructField("tpep_dropoff_datetime", TimestampNTZType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("DOLocationID", IntegerType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("Airport_fee", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True),
])

# 3. Lecture de TOUS les fichiers d'un coup, schéma imposé (wildcard)
#    (le motif ne ramasse que les 4 fichiers yellow_tripdata, pas taxi_zone_lookup.csv)
df_raw = (spark.read
    .format("parquet")
    .schema(taxi_schema)
    .load(f"{raw_path}/yellow_tripdata_*.parquet"))

# 4. Ajout des métadonnées de traçabilité
df_bronze = (df_raw
    .withColumn("ingestion_date", current_timestamp())
    .withColumn("source_file", input_file_name()))

# 5. Écriture en table Delta, par chemin (pas de saveAsTable en local)
(df_bronze.write
    .format("delta")
    .mode("overwrite")
    .save("../data/bronze/bronze_trips"))

print("Bronze écrit dans ../data/bronze/bronze_trips")

Bronze écrit dans ../data/bronze/bronze_trips


### Question 2 : inférence pour explorer, schéma manuel pour ingérer
Ce n'est pas une contradiction, car les deux étapes n'ont pas le même objectif.

Pendant l'exploration, on ne connaît pas encore les données. Laisser Spark deviner le schéma est le moyen le plus rapide de découvrir les colonnes, leurs types et leurs valeurs possiblement nulles, fichier par fichier. On accepte qu'il se trompe ou qu'il soit lent, puisqu'on ne produit rien.

Pendant l'ingestion, on veut au contraire un pipeline fiable et reproductible, et l'inférence y pose trois problèmes :
1. Elle peut donner un résultat différent si un nouveau fichier arrive avec un type différent (par exemple `passenger_count` en `double` une année et en `long` une autre). Les étapes suivantes cassent alors sans prévenir.
2. Elle coûte une lecture supplémentaire des données pour deviner les types, ce qui pèse sur 15,7 millions de lignes.
3. Elle ne garantit pas le type qu'on veut réellement utiliser.

Le `StructType` manuel fige un contrat : si un fichier ne le respecte pas, la lecture échoue franchement avec une erreur de conversion Parquet, au lieu de laisser passer une donnée mal typée. L'inférence sert donc à écrire ce contrat, puis le contrat la remplace.

## Étape 5 : Validation

On relit la table Delta depuis le disque pour prouver qu'elle existe et qu'elle contient les deux colonnes de traçabilité. Le `groupBy("source_file").count()` doit afficher 4 lignes, une par mois : preuve que le wildcard a ramassé les 4 fichiers et que la traçabilité fonctionne.

In [41]:
df_check = spark.read.format("delta").load("../data/bronze/bronze_trips")
df_check.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|      ingestion_date|         source_file|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------------+--------------------+
|   

In [42]:
# Comptage par fichier source : 4 lignes attendues, une par mois
df_check.groupBy("source_file").count().orderBy("source_file").show(truncate=False)

print("Nombre total de trajets en Bronze :", df_check.count())

+-------------------------------------------------------------------------+-------+
|source_file                                                              |count  |
+-------------------------------------------------------------------------+-------+
|file:///home/ilyas/tp_spark_core/data/raw/yellow_tripdata_2025-01.parquet|3475226|
|file:///home/ilyas/tp_spark_core/data/raw/yellow_tripdata_2025-04.parquet|3970553|
|file:///home/ilyas/tp_spark_core/data/raw/yellow_tripdata_2025-07.parquet|3898963|
|file:///home/ilyas/tp_spark_core/data/raw/yellow_tripdata_2025-10.parquet|4428699|
+-------------------------------------------------------------------------+-------+

Nombre total de trajets en Bronze : 15773441


### Question 3 : commentaire des quatre comptages mensuels

| Mois 2025 | Trajets | Jours | Trajets par jour |
|---|---|---|---|
| Janvier | 3 475 226 | 31 | ≈ 112 100 |
| Avril | 3 970 553 | 30 | ≈ 132 350 |
| Juillet | 3 898 963 | 31 | ≈ 125 770 |
| Octobre | 4 428 699 | 31 | ≈ 142 860 |
| **Total** | **15 773 441** | | |

Les quatre comptages additionnés donnent exactement le total de la table (15 773 441) : le wildcard a ramassé les quatre fichiers, sans perte ni doublon d'ingestion.

Octobre est le mois le plus chargé et janvier le moins chargé, avec un écart de 953 473 trajets, soit environ +27 %. L'écart est important mais ne me surprend pas. Janvier est traditionnellement un mois creux à New York (après les fêtes, il fait froid et il y a moins de tourisme), alors qu'octobre combine la pleine reprise de l'activité professionnelle et une saison touristique forte. Ramené au nombre de jours, avril dépasse juillet : on peut supposer qu'une partie des New-Yorkais quitte la ville l'été, ce qui fait baisser la demande malgré les touristes. Ces explications sont des hypothèses : nos données montrent l'écart, pas sa cause.

Chaque fichier contient quelques trajets dont la date de départ n'appartient pas au mois du fichier : 22 en janvier, 7 en avril, 7 en juillet et 13 en octobre, soit 49 au total (voir les mesures complémentaires). C'est négligeable (moins de 0,001 %), mais ce sont des dates incohérentes (horloge du taximètre mal réglée, course rattachée au mauvais mois) qu'il faudra traiter comme des anomalies en Séance 2.

Enfin, environ 16 millions de trajets sur quatre mois, c'est bien en dessous de 2017 et 2019, que le TP annonce deux à trois fois plus lourdes. Les taxis jaunes ont perdu une grande partie de leur marché au profit des VTC (Uber, Lyft), puis avec le Covid. Ces comptages serviront de point de comparaison avec les autres années.

## Couche Silver

Entrée : 1 ligne = 1 trajet brut (Bronze). Sortie : 1 ligne = 1 trajet propre et enrichi (Silver).
**Le grain ne change pas** : on nettoie la structure et on ajoute du contexte géographique, on ne regroupe rien et on ne filtre pas encore les valeurs aberrantes (Séance 2).

### 1. Lecture de la couche Bronze

In [43]:
from pyspark.sql.functions import col, broadcast

df_bronze_read = spark.read.format("delta").load("../data/bronze/bronze_trips")
nb_bronze = df_bronze_read.count()
print("Lignes en Bronze :", nb_bronze)

Lignes en Bronze : 15773441


### 2. Nettoyage structurel

- Les dates sont déjà en `TimestampNTZType` depuis le Bronze : on vérifie, et on n'y touche pas (pas de `to_timestamp`, qui décalerait les heures).
- Les montants (`*_amount`, `extra`, `mta_tax`, surcharges, frais) et `trip_distance` sont castés en `Double` explicitement : c'est le contrat de typage du Silver.
- Suppression des doublons complets avec `dropDuplicates()`, avec comptage avant / après.

In [44]:
from pyspark.sql.types import TimestampNTZType, DoubleType

# Vérification des dates (TimestampNTZType attendu)
for c in ["tpep_pickup_datetime", "tpep_dropoff_datetime"]:
    t = df_bronze_read.schema[c].dataType
    print(f"{c} : {t.simpleString()}", "OK" if isinstance(t, TimestampNTZType) else "A CORRIGER")

# Casting explicite des montants et de la distance en Double
colonnes_double = ["trip_distance", "fare_amount", "extra", "mta_tax", "tip_amount",
                   "tolls_amount", "improvement_surcharge", "total_amount",
                   "congestion_surcharge", "Airport_fee", "cbd_congestion_fee"]

df_typed = df_bronze_read
for c in colonnes_double:
    df_typed = df_typed.withColumn(c, col(c).cast("double"))

for c in colonnes_double:
    print(f"{c} : {df_typed.schema[c].dataType.simpleString()}")

tpep_pickup_datetime : timestamp_ntz OK
tpep_dropoff_datetime : timestamp_ntz OK
trip_distance : double
fare_amount : double
extra : double
mta_tax : double
tip_amount : double
tolls_amount : double
improvement_surcharge : double
total_amount : double
congestion_surcharge : double
Airport_fee : double
cbd_congestion_fee : double


**Typage et valeurs nulles.** Les montants et la distance étaient déjà en `double` dans le Bronze, grâce au `StructType` de l'ingestion. Le cast explicite est donc sans effet sur les valeurs : il ne crée aucun `null`. Un cast ne produit des `null` que lorsqu'une valeur ne peut pas être convertie dans le type cible (par exemple le texte `"abc"` casté en `double`). Le schéma strict appliqué dès le Bronze nous protège de ce cas. Le cast est conservé comme contrat de typage explicite de la couche Silver.

À distinguer : la source contient déjà des `null` qui ne viennent pas du cast. **3 315 521 trajets (21,0 %)** n'ont ni `passenger_count`, ni `RatecodeID`, ni `store_and_fwd_flag`, ni `congestion_surcharge`, ni `Airport_fee` (ce sont les mêmes lignes à chaque fois, voir les mesures complémentaires en fin de section). On les conserve : juger la qualité métier, c'est la Séance 2.

In [45]:
# Suppression des doublons complets
nb_avant_dedup = df_typed.count()
df_clean = df_typed.dropDuplicates()
nb_apres_dedup = df_clean.count()

print("Avant dropDuplicates :", nb_avant_dedup)
print("Après dropDuplicates :", nb_apres_dedup)
print("Doublons supprimés   :", nb_avant_dedup - nb_apres_dedup)

26/11/24 11:54:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:54:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:54:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:54:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:54:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:54:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:54:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:54:37 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:54:37 WARN RowBasedKeyValueBatch: Calling spill() on

Avant dropDuplicates : 15773441
Après dropDuplicates : 15773440
Doublons supprimés   : 1


### Question 5 : combien de doublons exacts, et d'où viennent-ils ?
Sur 15 773 441 trajets en Bronze, `dropDuplicates` n'a supprimé qu'**un seul** doublon exact (15 773 440 lignes après). C'est négligeable en volume (moins d'une ligne sur 15 millions), mais cela montre que la source n'est pas parfaitement propre et que l'étape de dédoublonnage n'est pas superflue.

**La ligne en double** (affichée dans les mesures complémentaires en fin de section) apparaît deux fois dans le fichier de **juillet** :
- `VendorID` 2, départ le 2025-07-05 à 17:19:29, arrivée 5 secondes plus tard ;
- départ et arrivée à JFK Airport (`PULocationID` = `DOLocationID` = 132), distance 0,0 mile ;
- `fare_amount` 3,0 $, `total_amount` 6,25 $, paiement en espèces (`payment_type` 2), `store_and_fwd_flag` = `N`.

**Ce n'est pas mon pipeline qui l'a créée** : en dédoublonnant sur les seules 20 colonnes métier (sans `ingestion_date` ni `source_file`), je trouve aussi exactement 1 doublon, et toutes les lignes ont la même `ingestion_date` (une seule écriture). Le doublon est donc déjà présent dans le fichier source.

Origines possibles :
- un envoi en double par le système du fournisseur (`VendorID` 2) ou par le taximètre, par exemple une transmission relancée après une erreur. En revanche, la piste du renvoi différé (`store_and_fwd_flag = Y`, course stockée faute de connexion puis renvoyée) est écartée ici, puisque la ligne porte `N` ;
- une erreur lors de la consolidation des fichiers mensuels par la TLC.

La course elle-même est suspecte (5 secondes, 0 mile, même zone au départ et à l'arrivée) : c'est probablement une course annulée ou un test, que les règles de qualité de la Séance 2 devraient rejeter. Ce sont des hypothèses : avec un seul cas, on ne peut pas trancher sur la cause.

### 3. Enrichissement : double jointure avec la dimension Zones (départ ET arrivée)

- Le CSV des zones est lu avec un **schéma explicite** : sans lui, `LocationID` serait lu comme une chaîne et la jointure avec `PULocationID` (entier) ne trouverait rien, sans erreur.
- On prépare **deux versions renommées** de la dimension, une par rôle (`zones_pu` pour le départ, `zones_do` pour l'arrivée), pour éviter l'erreur `AMBIGUOUS_REFERENCE`.
- Jointures `left` (on garde tous les trajets, même sans zone) et `broadcast` (la petite table Zones est copiée sur chaque Executor, pas de shuffle des trajets).

In [46]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

zones_schema = StructType([
    StructField("LocationID", IntegerType(), True),
    StructField("Borough", StringType(), True),
    StructField("Zone", StringType(), True),
    StructField("service_zone", StringType(), True),
])

df_zones = (spark.read
    .format("csv")
    .option("header", "true")
    .schema(zones_schema)
    .load("../data/raw/taxi_zone_lookup.csv"))

print("Nombre de zones :", df_zones.count())
df_zones.printSchema()

# Une version de la dimension par rôle
zones_pu = (df_zones
    .select("LocationID", "Borough", "Zone")
    .withColumnRenamed("LocationID", "PULocationID")
    .withColumnRenamed("Borough", "pickup_borough")
    .withColumnRenamed("Zone", "pickup_zone"))

zones_do = (df_zones
    .select("LocationID", "Borough", "Zone")
    .withColumnRenamed("LocationID", "DOLocationID")
    .withColumnRenamed("Borough", "dropoff_borough")
    .withColumnRenamed("Zone", "dropoff_zone"))

Nombre de zones : 265
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [47]:
# Double jointure Broadcast (left)
df_silver = (df_clean
    .join(broadcast(zones_pu), on="PULocationID", how="left")
    .join(broadcast(zones_do), on="DOLocationID", how="left"))

nb_apres_jointure = df_silver.count()
print("Avant jointure :", nb_apres_dedup)
print("Après jointure :", nb_apres_jointure)

# Preuve du Broadcast : le plan doit contenir BroadcastHashJoin (et non SortMergeJoin)
df_silver.explain()

26/11/24 11:54:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:54:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:54:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:54:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:54:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:54:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:54:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:54:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:54:51 WARN RowBasedKeyValueBatch: Calling spill() on

Avant jointure : 15773440
Après jointure : 15773440
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [DOLocationID#17657, PULocationID#17656, VendorID#17649, tpep_pickup_datetime#17650, tpep_dropoff_datetime#17651, passenger_count#17652L, trip_distance#17653, RatecodeID#17654L, store_and_fwd_flag#17655, payment_type#17658L, fare_amount#17659, extra#17660, mta_tax#17661, tip_amount#17662, tolls_amount#17663, improvement_surcharge#17664, total_amount#17665, congestion_surcharge#17666, Airport_fee#17667, cbd_congestion_fee#17668, ingestion_date#17669, source_file#17670, pickup_borough#18213, pickup_zone#18217, ... 2 more fields]
   +- BroadcastHashJoin [DOLocationID#17657], [DOLocationID#18224], LeftOuter, BuildRight, false
      :- Project [PULocationID#17656, VendorID#17649, tpep_pickup_datetime#17650, tpep_dropoff_datetime#17651, passenger_count#17652L, trip_distance#17653, RatecodeID#17654L, store_and_fwd_flag#17655, DOLocationID#17657, payment_type#17658L, fare_amou

### Question 4 : pourquoi la version naïve lève `AMBIGUOUS_REFERENCE`, et comment je l'ai réglé
La dimension zones doit être jointe deux fois : une fois sur `PULocationID` (départ), une fois sur `DOLocationID` (arrivée). Dans la version naïve, on joint deux fois la même table `df_zones` sans la modifier. Le résultat contient alors deux colonnes `LocationID`, deux `Borough` et deux `Zone`. Dès qu'on fait référence à `Borough`, Spark ne sait pas s'il s'agit de l'arrondissement de départ ou d'arrivée, et lève `AMBIGUOUS_REFERENCE`.

Ma préparation des dimensions règle le problème en créant **une version de la table par rôle** :
- `zones_pu` renomme les colonnes en `PULocationID`, `pickup_borough` et `pickup_zone` ;
- `zones_do` renomme les colonnes en `DOLocationID`, `dropoff_borough` et `dropoff_zone`.

Chaque colonne porte ainsi un nom unique qui dit son rôle. Et comme la jointure se fait avec `on="PULocationID"` puis `on="DOLocationID"`, la clé de jointure n'est pas dupliquée non plus.

**Pourquoi `broadcast` :** la dimension ne fait que 265 lignes. Plutôt que de redistribuer (shuffler) les 15,7 millions de trajets pour les aligner avec les zones, Spark envoie une copie de la petite table à chaque tâche. Le plan d'exécution affiché par `explain()` le confirme avec des `BroadcastHashJoin` au lieu de `SortMergeJoin`.

**Pourquoi `left` :** on garde tous les trajets, même ceux dont la zone serait introuvable. Le nombre de lignes est identique avant et après la jointure (15 773 440), donc le grain est préservé.

### 4. Stockage au format Delta

In [48]:
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .save("../data/silver/silver_trips"))

print("Silver écrit dans ../data/silver/silver_trips")

26/11/24 11:55:04 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:04 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:04 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:04 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:04 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:04 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:04 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:04 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:04 WARN RowBasedKeyValueBatch: Calling spill() on

Silver écrit dans ../data/silver/silver_trips


### 5. Validation

In [49]:
df_silver_check = spark.read.format("delta").load("../data/silver/silver_trips")

# Types corrects
df_silver_check.printSchema()

root
 |-- DOLocationID: integer (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- ingestion_date: timestamp (nullable = true)
 |-- source_file: string (n

In [50]:
# 5 trajets avec leurs 4 colonnes géographiques
(df_silver_check
    .select("tpep_pickup_datetime", "PULocationID", "pickup_borough", "pickup_zone",
            "DOLocationID", "dropoff_borough", "dropoff_zone", "total_amount")
    .show(5, truncate=False))

+--------------------+------------+--------------+---------------------+------------+---------------+---------------------------+------------+
|tpep_pickup_datetime|PULocationID|pickup_borough|pickup_zone          |DOLocationID|dropoff_borough|dropoff_zone               |total_amount|
+--------------------+------------+--------------+---------------------+------------+---------------+---------------------------+------------+
|2025-07-01 00:09:22 |211         |Manhattan     |SoHo                 |97          |Brooklyn       |Fort Greene                |25.75       |
|2025-07-01 00:43:37 |138         |Queens        |LaGuardia Airport    |65          |Brooklyn       |Downtown Brooklyn/MetroTech|63.07       |
|2025-07-01 00:58:09 |132         |Queens        |JFK Airport          |265         |N/A            |Outside of NYC             |83.44       |
|2025-07-01 00:24:18 |237         |Manhattan     |Upper East Side South|125         |Manhattan      |Hudson Sq                  |27.65       |

In [51]:
# Comptages avant / après
print("Bronze                    :", nb_bronze)
print("Avant dropDuplicates      :", nb_avant_dedup)
print("Après dropDuplicates      :", nb_apres_dedup)
print("Après jointure (Silver)   :", df_silver_check.count())

Bronze                    : 15773441
Avant dropDuplicates      : 15773441
Après dropDuplicates      : 15773440
Après jointure (Silver)   : 15773440


**Confirmation du grain.** `silver_trips` : **1 ligne = 1 trajet**, propre et enrichi. Le nombre de lignes passe de 15 773 441 (Bronze) à 15 773 440 (après suppression d'un doublon exact), puis reste à 15 773 440 après la double jointure. La jointure n'a ni dupliqué ni perdu de trajets. Aucun regroupement n'a été fait : le grain du Bronze est conservé dans le Silver.

| Étape | Lignes |
|---|---|
| Bronze | 15 773 441 |
| Après `dropDuplicates` | 15 773 440 (−1 doublon) |
| Après double jointure (Silver) | 15 773 440 (inchangé) |

Le changement de grain (1 ligne = 1 arrondissement) viendra en Séance 2 avec `gold_borough`.

In [52]:
# Trajets dont l'arrondissement de départ est null ou inconnu, et LocationID concernés
# Dans taxi_zone_lookup.csv : 264 -> Borough "Unknown", 265 -> Borough "N/A" (Outside of NYC)
df_pu_inconnu = df_silver_check.filter(
    col("pickup_borough").isNull() | col("pickup_borough").isin("Unknown", "N/A"))

print("Trajets avec arrondissement de départ null ou inconnu :", df_pu_inconnu.count())

(df_pu_inconnu
    .groupBy("PULocationID", "pickup_borough", "pickup_zone")
    .count()
    .orderBy(col("count").desc())
    .show(truncate=False))

Trajets avec arrondissement de départ null ou inconnu : 35420
+------------+--------------+--------------+-----+
|PULocationID|pickup_borough|pickup_zone   |count|
+------------+--------------+--------------+-----+
|264         |Unknown       |N/A           |28202|
|265         |N/A           |Outside of NYC|7218 |
+------------+--------------+--------------+-----+



### Question 6 : quels `LocationID` donnent un arrondissement null ou inconnu ?
Aucun trajet n'a de `pickup_borough` à `null` : tous les `PULocationID` de 2025 existent dans `taxi_zone_lookup.csv`. En revanche, **35 420 trajets** (environ 0,22 % du Silver) ont un arrondissement de départ inconnu :

| PULocationID | Borough | Zone | Trajets |
|---|---|---|---|
| 264 | Unknown | N/A | 28 202 |
| 265 | N/A | Outside of NYC | 7 218 |

Ces identifiants ne sont pas de vraies zones géographiques mais des codes réservés par la TLC : **264** signale une position de départ non enregistrée ou non identifiable, et **265** une prise en charge hors de New York.

**Ce que je décide d'en faire :** je les conserve dans `silver_trips`, car on ne supprime aucune ligne en Silver et le grain « 1 ligne = 1 trajet » doit rester intact. En revanche, ils ne peuvent pas être rattachés à un arrondissement. En Séance 2, je les enverrai en quarantaine avec le motif « zone de départ inconnue », pour qu'ils ne faussent pas les statistiques de `gold_borough`.

Côté arrivée, **106 476 trajets** (environ 0,68 %) ont un arrondissement inconnu : 36 540 en 264 (`Unknown`) et **69 936 en 265** (`Outside of NYC`). Le code 265 est presque dix fois plus fréquent à l'arrivée qu'au départ (69 936 contre 7 218). C'est logique : un taxi jaune peut déposer un client hors de New York (New Jersey, banlieue), mais n'a en principe pas le droit d'y prendre des clients. Ces trajets restent eux aussi dans le Silver, et leur traitement sera décidé en Séance 2 selon l'analyse (départ ou arrivée).

### Mesures complémentaires (utilisées dans les questions rédigées)

In [56]:
from pyspark.sql import functions as F

# Arrivées null ou inconnues (Question 6)
df_do_inconnu = df_silver_check.filter(
    col("dropoff_borough").isNull() | col("dropoff_borough").isin("Unknown", "N/A"))
print("Trajets avec arrondissement d'arrivée null ou inconnu :", df_do_inconnu.count())
df_do_inconnu.groupBy("DOLocationID", "dropoff_borough", "dropoff_zone").count().orderBy("DOLocationID").show(truncate=False)

# null générés par le cast (attendu : 0) et null déjà présents dans la source
df_silver_check.select([F.sum(col(c).isNull().cast("int")).alias(c)
                        for c in ["trip_distance", "fare_amount", "total_amount",
                                  "passenger_count", "RatecodeID", "Airport_fee",
                                  "store_and_fwd_flag", "congestion_surcharge"]]).show()

Trajets avec arrondissement d'arrivée null ou inconnu : 106476
+------------+---------------+--------------+-----+
|DOLocationID|dropoff_borough|dropoff_zone  |count|
+------------+---------------+--------------+-----+
|264         |Unknown        |N/A           |36540|
|265         |N/A            |Outside of NYC|69936|
+------------+---------------+--------------+-----+

+-------------+-----------+------------+---------------+----------+-----------+------------------+--------------------+
|trip_distance|fare_amount|total_amount|passenger_count|RatecodeID|Airport_fee|store_and_fwd_flag|congestion_surcharge|
+-------------+-----------+------------+---------------+----------+-----------+------------------+--------------------+
|            0|          0|           0|        3315521|   3315521|    3315521|           3315521|             3315521|
+-------------+-----------+------------+---------------+----------+-----------+------------------+--------------------+



In [54]:
# Le doublon exact : quelle ligne, quel fichier ? (Question 5)
(df_bronze_read
    .groupBy(*df_bronze_read.columns).count()
    .filter("count > 1")
    .show(truncate=False, vertical=True))

# Contrôle : doublons sur les seules colonnes métier (sans ingestion_date ni source_file)
colonnes_metier = [c for c in df_bronze_read.columns if c not in ("ingestion_date", "source_file")]
print("Doublons sur les colonnes métier :", nb_bronze - df_bronze_read.dropDuplicates(colonnes_metier).count())
print("Valeurs distinctes de ingestion_date :", df_bronze_read.select("ingestion_date").distinct().count())

26/11/24 11:55:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:39 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:39 WARN RowBasedKeyValueBatch: Calling spill() on

-RECORD 0------------------------------------------------------------------------------------------
 VendorID              | 2                                                                         
 tpep_pickup_datetime  | 2025-07-05 17:19:29                                                       
 tpep_dropoff_datetime | 2025-07-05 17:19:34                                                       
 passenger_count       | 1                                                                         
 trip_distance         | 0.0                                                                       
 RatecodeID            | 1                                                                         
 store_and_fwd_flag    | N                                                                         
 PULocationID          | 132                                                                       
 DOLocationID          | 132                                                                       


26/11/24 11:55:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/24 11:55:54 WARN RowBasedKeyValueBatch: Calling spill() on

Doublons sur les colonnes métier : 1
Valeurs distinctes de ingestion_date : 1


In [55]:
# Trajets dont la date de départ n'est pas dans le mois du fichier (Question 3)
(df_bronze_read
    .withColumn("mois_fichier", F.regexp_extract("source_file", r"2025-(\d\d)", 1).cast("int"))
    .withColumn("hors_mois", (F.month("tpep_pickup_datetime") != col("mois_fichier"))
                             | (F.year("tpep_pickup_datetime") != 2025))
    .groupBy("mois_fichier")
    .agg(F.count("*").alias("trajets"),
         F.sum(col("hors_mois").cast("int")).alias("trajets_hors_mois"))
    .orderBy("mois_fichier")
    .show())

+------------+-------+-----------------+
|mois_fichier|trajets|trajets_hors_mois|
+------------+-------+-----------------+
|           1|3475226|               22|
|           4|3970553|                7|
|           7|3898963|                7|
|          10|4428699|               13|
+------------+-------+-----------------+

